# Analyzing the Impact of Weather on Public Transportation - Indian Cities
## Exploratory Data Analysis & Data Engineering Pipeline

**Objective:** Analyze how weather conditions affect public transportation delays across different transport modes in Indian cities.

**Data Sources:**
- Weather Data: OpenWeatherMap API / IMD (India Meteorological Department)
- Transportation Data: Delhi Metro, Mumbai Local, Bangalore BMTC, Kolkata Metro

**Cities Covered:** Delhi, Mumbai, Bangalore, Kolkata, Chennai

**Tech Stack:** PySpark, PostgreSQL, Pandas, Seaborn, Plotly

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, when, to_date
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import requests
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 2. Initialize Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("IndiaWeatherTransportAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## 3. Data Ingestion
### 3.1 Weather Data (Indian Cities)

In [ ]:
# OpenWeatherMap API (Free tier: current + 5 day forecast)
# For historical: Use Visual Crossing Weather API or IMD data
OWM_API_KEY = "YOUR_API_KEY"  # Get from: https://openweathermap.org/api

INDIAN_CITIES = {
    'Delhi': {'lat': 28.6139, 'lon': 77.2090},
    'Mumbai': {'lat': 19.0760, 'lon': 72.8777},
    'Bangalore': {'lat': 12.9716, 'lon': 77.5946},
    'Kolkata': {'lat': 22.5726, 'lon': 88.3639},
    'Chennai': {'lat': 13.0827, 'lon': 80.2707}
}

def fetch_weather_data(api_key, city_name, lat, lon):
    # Visual Crossing Weather API (free 1000 records/day)
    url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/{lat},{lon}/2023-01-01/2023-12-31"
    params = {'key': api_key, 'unitGroup': 'metric', 'include': 'days'}
    response = requests.get(url, params=params)
    return response.json()

# Alternative: IMD Data Portal
# Download from: https://www.imdpune.gov.in/ or https://data.gov.in/

# Load pre-downloaded weather data
weather_df = spark.read.csv("data/raw/india_weather_data.csv", header=True, inferSchema=True)
weather_df.show(5)

### 3.2 Transportation Data (Indian Transit Systems)

In [ ]:
# Indian Transportation Data Sources:
# 1. Delhi Metro: https://otd.delhi.gov.in/data/ (Open Transit Data)
# 2. Mumbai Local: https://data.gov.in/ (search 'Mumbai railway')
# 3. Bangalore BMTC: https://www.mybmtc.com/ or https://transitfeeds.com/
# 4. Kolkata Metro: https://www.kmcgov.in/KMCPortal/
# 5. Chennai Metro: https://chennaimetrorail.org/

# Load GTFS data
routes_df = spark.read.csv("data/raw/india_gtfs/routes.txt", header=True, inferSchema=True)
trips_df = spark.read.csv("data/raw/india_gtfs/trips.txt", header=True, inferSchema=True)
stop_times_df = spark.read.csv("data/raw/india_gtfs/stop_times.txt", header=True, inferSchema=True)

# Performance data (Delhi Metro publishes monthly reports)
delays_df = spark.read.csv("data/raw/india_delays_performance.csv", header=True, inferSchema=True)

print(f"Routes: {routes_df.count()}, Delays: {delays_df.count()}")
delays_df.show(5)

## 4. Data Cleaning & Transformation

### 4.1 Clean Weather Data

In [ ]:
weather_clean = weather_df \
    .withColumn("date", to_date(col("date"))) \
    .fillna({"precipitation": 0, "wind_speed": 0, "humidity": 0}) \
    .withColumn("avg_temp", (col("temp_max") + col("temp_min")) / 2)

# Categorize weather (Indian context: monsoon, heat waves, no snow)
weather_clean = weather_clean.withColumn(
    "weather_condition",
    when(col("precipitation") > 50, "Heavy Rain")  # mm
    .when(col("precipitation") > 10, "Moderate Rain")
    .when(col("precipitation") > 2, "Light Rain")
    .when(col("avg_temp") > 40, "Extreme Heat")  # Celsius
    .when(col("humidity") > 85, "High Humidity")
    .otherwise("Clear")
)

# Add monsoon season flag (June-September)
from pyspark.sql.functions import month
weather_clean = weather_clean.withColumn(
    "is_monsoon",
    when(month(col("date")).isin([6, 7, 8, 9]), 1).otherwise(0)
)

weather_clean.groupBy("weather_condition").count().show()

### 4.2 Clean Transportation Data

In [ ]:
transport_clean = delays_df \
    .withColumn("date", to_date(col("timestamp"))) \
    .withColumn("delay_minutes", col("delay_seconds") / 60) \
    .filter(col("delay_minutes").isNotNull())

transport_clean = transport_clean.join(routes_df, "route_id", "left")

daily_delays = transport_clean.groupBy("date", "route_id", "route_name", "transport_type", "city") \
    .agg(
        avg("delay_minutes").alias("avg_delay_minutes"),
        count("*").alias("total_trips")
    )

daily_delays.show(5)

### 4.3 Join Weather & Transportation Data

In [ ]:
fact_table = daily_delays.join(weather_clean, ["date", "city"], "inner")

fact_table = fact_table.select(
    "date", "city", "route_id", "route_name", "transport_type",
    "avg_delay_minutes", "total_trips",
    "precipitation", "avg_temp", "wind_speed", "humidity",
    "weather_condition", "is_monsoon"
)

print(f"Total records: {fact_table.count()}")
fact_table.show(5)

## 5. Exploratory Data Analysis

### 5.1 Data Overview

In [ ]:
df = fact_table.toPandas()
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nSummary Statistics:")
df.describe()

### 5.2 Correlation Analysis

In [ ]:
corr_cols = ['avg_delay_minutes', 'precipitation', 'avg_temp', 'wind_speed', 'humidity']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Correlation: Weather vs Delays (Indian Cities)', fontsize=16)
plt.tight_layout()
plt.show()

print(f"Precipitation vs Delay: {df['precipitation'].corr(df['avg_delay_minutes']):.3f}")
print(f"Temperature vs Delay: {df['avg_temp'].corr(df['avg_delay_minutes']):.3f}")
print(f"Humidity vs Delay: {df['humidity'].corr(df['avg_delay_minutes']):.3f}")

### 5.3 Monsoon Impact Analysis

In [ ]:
monsoon_impact = df.groupby('is_monsoon')['avg_delay_minutes'].agg(['mean', 'median', 'count'])
monsoon_impact.index = ['Non-Monsoon', 'Monsoon']
print("Monsoon vs Non-Monsoon Delays:")
print(monsoon_impact)

fig, ax = plt.subplots(figsize=(10, 6))
monsoon_impact['mean'].plot(kind='bar', color=['#2ca02c', '#d62728'], ax=ax)
plt.title('Average Delay: Monsoon vs Non-Monsoon', fontsize=16)
plt.ylabel('Average Delay (minutes)', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5.4 City-wise Analysis

In [ ]:
city_weather = df.groupby(['city', 'weather_condition'])['avg_delay_minutes'].mean().reset_index()

fig = px.bar(city_weather, x='city', y='avg_delay_minutes', 
             color='weather_condition', barmode='group',
             title='Average Delay by City and Weather Condition',
             labels={'avg_delay_minutes': 'Average Delay (minutes)', 'city': 'City'})
fig.show()

### 5.5 Transport Type Analysis

In [ ]:
transport_weather = df.groupby(['transport_type', 'weather_condition'])['avg_delay_minutes'].mean().reset_index()

fig = px.bar(transport_weather, x='transport_type', y='avg_delay_minutes', 
             color='weather_condition', barmode='group',
             title='Average Delay by Transport Type and Weather (Indian Cities)',
             labels={'avg_delay_minutes': 'Average Delay (minutes)', 
                    'transport_type': 'Transport Type'})
fig.show()

### 5.6 Rainfall Threshold Analysis

In [ ]:
df['rain_category'] = pd.cut(df['precipitation'], 
                              bins=[0, 2, 10, 35, 65, 200], 
                              labels=['No Rain', 'Light', 'Moderate', 'Heavy', 'Very Heavy'])

rain_threshold = df.groupby('rain_category')['avg_delay_minutes'].agg(['mean', 'count'])
print("\nDelay by Rainfall Intensity (IMD Classification):")
print(rain_threshold)

fig = px.scatter(df[df['precipitation'] > 0], x='precipitation', y='avg_delay_minutes',
                 color='city', trendline='ols',
                 title='Rainfall vs Delay (Indian Cities)',
                 labels={'precipitation': 'Rainfall (mm)', 
                        'avg_delay_minutes': 'Average Delay (minutes)'})
fig.show()

### 5.7 Heat Wave Impact

In [ ]:
heat_impact = df[df['avg_temp'] > 35].groupby('city')['avg_delay_minutes'].mean().sort_values(ascending=False)
print("\nAverage Delay During High Temperature Days (>35°C):")
print(heat_impact)

fig = px.scatter(df, x='avg_temp', y='avg_delay_minutes', color='city',
                 title='Temperature vs Delay (Indian Cities)',
                 labels={'avg_temp': 'Temperature (°C)', 
                        'avg_delay_minutes': 'Average Delay (minutes)'})
fig.show()

### 5.8 Statistical Testing

In [ ]:
# T-test: Monsoon vs Non-Monsoon
monsoon_delays = df[df['is_monsoon'] == 1]['avg_delay_minutes']
non_monsoon_delays = df[df['is_monsoon'] == 0]['avg_delay_minutes']

t_stat, p_value = stats.ttest_ind(monsoon_delays, non_monsoon_delays)
print(f"T-test (Monsoon vs Non-Monsoon): t={t_stat:.3f}, p={p_value:.4f}")
print(f"Significant difference: {p_value < 0.05}")

# ANOVA: All weather conditions
groups = [df[df['weather_condition'] == cond]['avg_delay_minutes'] 
          for cond in df['weather_condition'].unique()]
f_stat, p_value = stats.f_oneway(*groups)
print(f"\nANOVA (All conditions): F={f_stat:.3f}, p={p_value:.4f}")

## 6. Key Insights & KPIs

In [ ]:
kpis = {
    'Avg Delay (Clear)': df[df['weather_condition'] == 'Clear']['avg_delay_minutes'].mean(),
    'Avg Delay (Monsoon)': df[df['is_monsoon'] == 1]['avg_delay_minutes'].mean(),
    'Avg Delay (Heavy Rain)': df[df['weather_condition'] == 'Heavy Rain']['avg_delay_minutes'].mean(),
    'Most Affected City': df.groupby('city')['avg_delay_minutes'].mean().idxmax(),
    'Most Affected Transport': df.groupby('transport_type')['avg_delay_minutes'].mean().idxmax(),
    'Worst Weather Type': df.groupby('weather_condition')['avg_delay_minutes'].mean().idxmax()
}

print("\n=== KEY PERFORMANCE INDICATORS (Indian Cities) ===")
for key, value in kpis.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f} minutes")
    else:
        print(f"{key}: {value}")

## 7. Data Export to PostgreSQL

In [ ]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'india_weather_transport',
    'user': 'postgres',
    'password': 'your_password'
}

engine = create_engine(
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

df.to_sql('fact_india_daily_delays', engine, if_exists='replace', index=False)
print("Data exported to PostgreSQL successfully!")

## 8. Recommendations for Indian Cities

**For City Planners:**
- Prioritize monsoon-resilient infrastructure (waterlogging prevention)
- Focus on bus routes (most weather-sensitive) for covered stops
- Implement heat-resistant measures for summer months

**For Transport Authorities:**
- Deploy additional buses during monsoon season (June-September)
- Adjust schedules when rainfall >35mm forecasted
- Pre-position maintenance crews before monsoon onset

**For Commuters:**
- Build "Commute Forecast" app with monsoon alerts
- Send notifications: "Expect 20-min delay on Route X due to heavy rain"

**Next Steps:**
- Deploy ML model (Random Forest) for delay prediction
- Integrate with Apache Airflow for daily pipeline
- Create Power BI/Tableau dashboard

## 9. Cleanup

In [ ]:
spark.stop()